In [ ]:
import numpy as np
import torch
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import time
from PIL import Image, ImageOps

In [ ]:
DATA_MAIN_FOLDER = ""
DATA_SAVE_FOLDER = ""

In [ ]:
def get_singular_values(A):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    A = torch.as_tensor(A, dtype=torch.float32, device=device)

    if not torch.isfinite(A).all():
        raise ValueError("A contains NaN or Inf values; clean it before computing singular values.")

    # Only the singular values are used below. A full SVD of this wide matrix
    # asks cuSOLVER for very large U/Vh factors and can fail in the default driver.
    try:
        if A.is_cuda:
            D = torch.linalg.svdvals(A, driver="gesvd")
        else:
            D = torch.linalg.svdvals(A)
    except RuntimeError as err:
        print(f"svdvals failed ({err}); falling back to eigenvalues of A @ A.T")
        gram = A @ A.mT
        D = torch.linalg.eigvalsh(gram).clamp_min_(0).sqrt().flip(0)

    return D

# Smallnorb

In [ ]:
_MAGIC_TO_DTYPE = {
    0x1E3D4C51: np.dtype("<f4"),
    0x1E3D4C52: np.dtype("|V1"),
    0x1E3D4C53: np.dtype("<f8"),
    0x1E3D4C54: np.dtype("<i4"),
    0x1E3D4C55: np.dtype("u1"),
    0x1E3D4C56: np.dtype("<i2"),
}


def read_norb_matrix(path, mmap_mode="r"):
    """
    Read a small NORB binary matrix file.

    Returns a NumPy array (or memmap) reshaped according to the dimensions
    encoded in the file header.
    """
    path = Path(path)

    with path.open("rb") as handle:
        magic = np.fromfile(handle, dtype="<i4", count=1)[0]
        ndim = int(np.fromfile(handle, dtype="<i4", count=1)[0])
        dims = np.fromfile(handle, dtype="<i4", count=max(ndim, 3))[:ndim]
        data_offset = handle.tell()

    try:
        dtype = _MAGIC_TO_DTYPE[int(magic)]
    except KeyError as exc:
        raise ValueError(f"Unsupported NORB magic number: {magic:#x}") from exc

    shape = tuple(int(dim) for dim in dims)

    if mmap_mode is None:
        with path.open("rb") as handle:
            handle.seek(data_offset)
            data = np.fromfile(handle, dtype=dtype, count=int(np.prod(shape)))
        return data.reshape(shape)

    return np.memmap(
        path,
        dtype=dtype,
        mode=mmap_mode,
        offset=data_offset,
        shape=shape,
    )


def load_smallnorb_first_images(path, dtype=None, mmap_mode="r"):
    """
    Return the first image from each stereo pair as a matrix of shape
    (24300, 9216) for the training file.
    """
    tensor = read_norb_matrix(path, mmap_mode=mmap_mode)

    if tensor.ndim != 4 or tensor.shape[1:] != (2, 96, 96):
        raise ValueError(
            "Expected a small NORB image tensor with shape "
            f"(n_samples, 2, 96, 96), got {tensor.shape}."
        )

    matrix = tensor[:, 0, :, :].reshape(tensor.shape[0], -1)
    if dtype is not None:
        matrix = matrix.astype(dtype, copy=False)
    return matrix


In [ ]:
data_path = Path(DATA_MAIN_FOLDER+'smallnorb-5x46789x9x18x6x2x96x96-training-dat.mat')
A = load_smallnorb_first_images(data_path, mmap_mode='c')
print(A.shape)
print(A.dtype)

In [ ]:
D = get_singular_values(A)
D

In [ ]:
np.savez_compressed(DATA_SAVE_FOLDER+"smallnorb.npz", A=A, D=D)

# Imagenet

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}


def _image_paths(folder, recursive=False):
    folder = Path(folder)
    iterator = folder.rglob("*") if recursive else folder.iterdir()
    return sorted(path for path in iterator if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)


def _load_red_float_array(path, target_size=None):
    with Image.open(path) as image:
        image = image.convert("RGB")
        if target_size is not None:
            image = ImageOps.fit(image, target_size, method=Image.Resampling.BICUBIC)
        return np.asarray(image, dtype=np.float32)[:, :, 0] / 255.0

def red_vectors_from_directory(
    data_dir,
    target_size=None,
    batch_size=256,
    num_workers=8,
    output_path=None,
    recursive=False,
):
    paths = _image_paths(data_dir, recursive=recursive)
    if not paths:
        raise ValueError(f"No image files found in {data_dir!r}.")

    chunks = []
    output = None
    start_time = time.perf_counter()

    for start in range(0, len(paths), batch_size):
        batch_paths = paths[start : start + batch_size]
        if num_workers and num_workers > 1:
            with ThreadPoolExecutor(max_workers=num_workers) as pool:
                arrays = list(pool.map(lambda path: _load_red_float_array(path, target_size), batch_paths))
        else:
            arrays = [_load_red_float_array(path, target_size) for path in batch_paths]

        try:
            batch = np.stack(arrays)
        except ValueError as exc:
            raise ValueError(
                "Images must have the same shape within each batch. "
                "Pass target_size=(width, height) to force a consistent size."
            ) from exc

        vectors = batch.reshape(len(batch_paths), -1).astype(np.float32, copy=False)

        if output_path is None:
            chunks.append(vectors)
        else:
            if output is None:
                output = np.lib.format.open_memmap(
                    output_path,
                    mode="w+",
                    dtype=np.float32,
                    shape=(len(paths), vectors.shape[1]),
                )
            output[start : start + len(batch_paths)] = vectors

    elapsed = time.perf_counter() - start_time
    print(f"processed {len(paths)} images in {elapsed:.2f} seconds")

    if output_path is not None:
        output.flush()
        return paths, output_path

    return paths, np.vstack(chunks)


In [ ]:
train_data_dir = Path("data_general/train")

train_paths, red_vectors = red_vectors_from_directory(
    train_data_dir,
    target_size=(224, 224),
    batch_size=256,
    num_workers=8,
    output_path=None,
    recursive=True,
)

print(f"processed train images: {len(train_paths)}")
print(f"red_vectors shape: {red_vectors.shape}")
print(f"matrix file: {red_vectors}")

In [ ]:
A = red_vectors
D = get_singular_values(A)

In [ ]:
np.savez_compressed(DATA_SAVE_FOLDER+"imagenet.npz", A=A, D=D)

# Polydecay

In [ ]:
def poly_decay_torch(num_rows, num_cols, rate):
    device = torch.device("cuda")
    dtype = torch.float32
    core_dim = min(num_cols, num_rows)
    max_dim = max(num_cols, num_rows)

    diag = max_dim / (torch.arange(1, core_dim + 1, dtype=dtype, device=device) ** rate)
    core = torch.diag(diag)

    U, _ = torch.linalg.qr(
        torch.rand((num_rows, core_dim), dtype=dtype, device=device),
        mode="reduced",
    )
    V, _ = torch.linalg.qr(
        torch.rand((num_cols, core_dim), dtype=dtype, device=device),
        mode="reduced",
    )

    return U @ core @ V.mT

In [ ]:
A = poly_decay_torch(30000 , 15000, 1)
A_np = A.detach().cpu().numpy()
D = 30000 / (np.arange(1, 15001, dtype=np.float32) ** 1)
A

In [ ]:
np.savez_compressed(DATA_SAVE_FOLDER+"polydecay.npz", A=A, D=D)
